# 08 Model Evaluation

Review training outputs, explainability artifacts, and drift readiness using the current pipeline artifacts.

## Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

import pandas as pd
from IPython.display import display, Image, HTML

sys.path.append(os.path.abspath('..'))
from src.utils import load_config
from src.drift_detector import DriftDetector
from src.schemas import ML_FEATURE_COLUMNS

config = load_config('../configs/config.yaml')
paths = config['paths']
processed_dir = Path('..') / paths['processed_data_dir']
models_dir = Path('..') / 'models'
logs_dir = Path('..') / 'logs'
outputs_dir = Path('..') / 'outputs'

ml_dataset_path = processed_dir / paths['ml_dataset_file']
comparison_path = logs_dir / 'model_comparison.json'
shap_values_path = outputs_dir / 'shap_values.parquet'


## Artifact Inventory

In [ ]:
artifact_paths = [
    comparison_path,
    outputs_dir / 'model_comparison.png',
    outputs_dir / 'roc_curves.png',
    outputs_dir / 'shap_summary.png',
    outputs_dir / 'drift_heatmap.png',
    logs_dir / 'drift_report.json',
    logs_dir / 'drift_summary.json',
    models_dir / 'feature_importance.json',
]
display(pd.DataFrame({
    'artifact': [str(path) for path in artifact_paths],
    'exists': [path.exists() for path in artifact_paths],
}))


## Model Comparison Report

In [ ]:
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
    print(f"Best model: {comparison.get('best_model')}")
    display(pd.DataFrame(comparison.get('ranking', [])))
else:
    print('No model comparison report exists yet. Run notebook 07 or `main.py --stage train --force`.')


## Visual Review

In [ ]:
for image_name in ['model_comparison.png', 'roc_curves.png', 'shap_summary.png', 'drift_heatmap.png']:
    image_path = outputs_dir / image_name
    if image_path.exists():
        print(image_name)
        display(Image(filename=str(image_path)))
    else:
        print(f'{image_name} not found yet.')


## SHAP Outputs

In [ ]:
if shap_values_path.exists():
    shap_df = pd.read_parquet(shap_values_path)
    print(f'SHAP values shape: {shap_df.shape}')
    display(shap_df.head())
else:
    print('No SHAP parquet found yet.')

for idx in range(3):
    html_path = outputs_dir / f'shap_force_plot_{idx}.html'
    if html_path.exists():
        print(f'Loaded {html_path.name}')
        display(HTML(html_path.read_text(encoding='utf-8')))
        break
else:
    print('No SHAP force-plot HTML files found yet.')


## Drift Check

In [ ]:
if not ml_dataset_path.exists():
    print('No final ML dataset found.')
else:
    df_ml = pd.read_parquet(ml_dataset_path)
    month_count = pd.to_datetime(df_ml['scheduled_dep'], utc=True, errors='coerce').dt.to_period('M').nunique() if 'scheduled_dep' in df_ml.columns else 0
    print(f'Month partitions detected: {month_count}')

    if month_count >= 2:
        detector = DriftDetector()
        drift_df = detector.monthly_drift_report(df_ml, feature_cols=ML_FEATURE_COLUMNS)
        if drift_df is not None and not drift_df.empty:
            display(drift_df.head())
            display(detector.stability_ranking(drift_df).head(15))
    else:
        print('Drift analysis is not meaningful yet because the dataset does not span multiple months.')


## Evaluation Notes

In [ ]:
notes = []
if ml_dataset_path.exists():
    df_ml = pd.read_parquet(ml_dataset_path)
    if 'label' in df_ml.columns:
        label_counts = df_ml['label'].value_counts(dropna=False)
        notes.append({'check': 'label_classes', 'value': int((label_counts > 0).sum()), 'status': 'ok' if int((label_counts > 0).sum()) >= 2 else 'blocked'})
    notes.append({'check': 'rows', 'value': len(df_ml), 'status': 'ok' if len(df_ml) > 0 else 'blocked'})

display(pd.DataFrame(notes))
